In [ ]:
import json

def load_json(filepath: str):
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

races_data = load_json("assets/usable/races.json")

def trace_types(data, path=""):
    current_type = type(data).__name__

    new_path = f"{path}->{current_type}" if path else current_type

    if isinstance(data, list):
        for item in data:
            yield from trace_types(item, new_path)
    elif isinstance(data, dict):
        for value in data.values():
            yield from trace_types(value, new_path)
    else:
        yield new_path

paths = []
for type_path in trace_types(races_data):
    paths.append(type_path)

paths = list(set(paths))

for p in paths:
    print(p)

list->dict->list->dict->dict->list->str
list->dict->dict->str
list->dict->list->dict->list->dict->list->str
list->dict->bool
list->dict->list->dict->list->str
list->dict->list->dict->bool
list->dict->list->dict->list->dict->str
list->dict->list->dict->list->dict->list->list->str
list->dict->list->dict->str
list->dict->str
list->dict->list->dict->dict->int
list->dict->int
list->dict->list->str
list->dict->dict->int
list->dict->list->dict->dict->dict->dict->list->str
list->dict->list->dict->int


```plaintext
list->dict->list->dict->dict->list->str
list->dict->dict->str
list->dict->list->dict->list->dict->list->str
list->dict->bool
list->dict->list->dict->list->str
list->dict->list->dict->bool
list->dict->list->dict->list->dict->str
list->dict->list->dict->list->dict->list->list->str
list->dict->list->dict->str
list->dict->str
list->dict->list->dict->dict->int
list->dict->int
list->dict->list->str
list->dict->dict->int
list->dict->list->dict->dict->dict->dict->list->str
list->dict->list->dict->int
```

In [ ]:
from math import floor
from pydantic import computed_field, BaseModel, Field, model_validator
from typing import Literal, Optional, Self


class Abilities(BaseModel):
    strength: int = 0
    dexterity: int = 0
    constitution: int = 0
    intelligence: int = 0
    wisdom: int = 0
    charisma: int = 0

    @computed_field
    @property
    def strength_mod(self) -> int:
        return floor(self.strength - 10 / 2)

    @computed_field
    @property
    def dexterity_mod(self) -> int:
        return floor(self.dexterity - 10 / 2)

    @computed_field
    @property
    def constitution_mod(self) -> int:
        return floor(self.constitution - 10 / 2)

    @computed_field
    @property
    def intelligence_mod(self) -> int:
        return floor(self.intelligence - 10 / 2)

    @computed_field
    @property
    def wisdom_mod(self) -> int:
        return floor(self.wisdom - 10 / 2)

    @computed_field
    @property
    def charisma_mod(self) -> int:
        return floor(self.charisma - 10 / 2)


class DiceMod(BaseModel):
    amount: int = Field(ge=1)
    sides: Literal[4, 6, 8, 10, 12, 20]


class HeightWeight(BaseModel):
    base_height: int
    height_mod: DiceMod
    base_weight: int
    weight_mod: DiceMod


class Age(BaseModel):
    mature: int
    maximum: int


class ChooseResistance(BaseModel):
    choice: Literal[
        "fire", "poison", "cold", "psychic", "magic", "acid", "lightning", "necrotic"
    ]
    fire: bool = False
    poison: bool = False
    cold: bool = False
    psychic: bool = False
    magic: bool = False
    acid: bool = False
    lightning: bool = False
    necrotic: bool = False

    @model_validator(mode="after")
    def set_resistance_choice(self) -> Self:
        setattr(self, self.choice, True)
        return self

"""
Acrobatics (Dex)
Animal Handling (Wis)
Arcana (Int)
Athletics (Str)
Deception (Cha)
History (Int)
Insight (Wis)
Intimidation (Cha)
Investigation (Int)
Medicine (Wis)
Nature (Int)
Perception (Wis)
Performance (Cha)
Persuasion (Cha)
Religion (Int)
Sleight of Hand (Dex)
Stealth (Dex)
Survival (Wis)
"""

class Race(BaseModel):
    name: str
    size: Literal["Small", "Medium", "Large"]
    speed: int = Field(ge=10, le=50)
    abilities: Abilities
    height_and_weight: HeightWeight
    age: Age
    language_proficiencies: list[str] = ["Common"]
    choose_resistance: ChooseResistance
    darkvision: Optional[int]
    tool_proficiencies: Optional[
        list[Literal["Smith's Tools", "Brewer's Supplies", "Mason's Tools"]]
    ]
    weapon_proficiencies: Optional[
        list[Literal["Battleaxe", "Handaxe", "Light Hammer", "Warhammer"]]
    ]
    skill_proficiencies: Optional[
        list[Literal["Acrobatics", "Animal Handling", "Arcana", ]]
    ]